# Medical Triage Agent AI POC
## Google Colab Training Pipeline


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Vérifications
print("GPU :", torch.cuda.get_device_name(0))   # doit afficher Tesla T4
print("VRAM totale :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("BF16 natif :", torch.cuda.is_bf16_supported())   # → False sur T4 ✓

In [2]:
from __future__ import annotations
import logging

logging.basicConfig(level=logging.INFO)
logger=logging.getLogger('training_pipeline')


# Section 2 — Bootstrap Runtime


In [ ]:
import sys
import subprocess
import os

UPDATED = False


def install(pkg, version=None):
    """Installe un package normalement (avec ses dépendances)."""
    global UPDATED

    if version:
        target = f"{pkg}=={version}"
    else:
        target = pkg

    print(f"📦 Installation : {target}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", target],
        check=True
    )

    UPDATED = True


def install_no_deps(pkg, version=None):
    """Installe un package SANS toucher aux dépendances déjà installées."""
    global UPDATED

    if version:
        target = f"{pkg}=={version}"
    else:
        target = pkg

    print(f"📦 Installation (--no-deps) : {target}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-U", "--no-deps", target],
        check=True
    )

    UPDATED = True


print("🚀 INSTALLATION DES DÉPENDANCES\n")

# --- 1. Dépendances de base (unsloth va potentiellement les downgrader) ---
install("accelerate", "1.14.0")
install("peft", "0.19.1")
install("datasets", "4.0.0")
install("bitsandbytes", "0.49.2")
install("huggingface_hub", "1.20.1")
install("wandb", "0.28.0")
install("evaluate", "0.4.6")
install("sentencepiece", "0.2.1")
install("safetensors", "0.8.0")

# --- 2. Unsloth, sans ses dépendances pour ne pas écraser les versions ci-dessus/ci-dessous ---
install_no_deps("unsloth")

# --- 3. Réinstallation EN DERNIER des packages critiques pour forcer les bonnes versions ---
install("torch", "2.11.0+cu128")
install("transformers", "5.12.1")
install("trl", "1.7.0")
install("torchao", "0.17.0")

print("\n✅ INSTALLATION TERMINÉE")

if UPDATED:
    print("🔄 REDÉMARRAGE OBLIGATOIRE DU RUNTIME")
    os.kill(os.getpid(), 9)

In [ ]:
# =======================================================
# Cellule 2 : 👉 À exécuter APRÈS redémarrage du runtime
# =======================================================

import torch
import transformers
import trl
import accelerate
import peft
import platform

print("=== ENVIRONMENT CHECK ===\n")

print("Python       :", platform.python_version())
print("Torch        :", torch.__version__)
print("Transformers :", transformers.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)
print("PEFT         :", peft.__version__)

print("\nCUDA :", torch.version.cuda)
print("GPU  :", torch.cuda.is_available())

print("\nTorch file   :", torch.__file__)
print("Transformers :", transformers.__file__)
print("TRL file     :", trl.__file__)

print("\n=== TORCH OPS CHECK ===")

print("Has _c10d_functional:", hasattr(torch.ops, "_c10d_functional"))

if hasattr(torch.ops, "_c10d_functional"):
    print("_wrap_tensor_autograd:",
          "_wrap_tensor_autograd" in dir(torch.ops._c10d_functional))

In [ ]:
#!pip show unsloth

In [ ]:
# =======================================================
# Cellule 3 : 👉 À exécuter uniquement si Cellule 2 OK
# =======================================================

from trl import DPOConfig, DPOTrainer

print("✅ TRL import OK")

In [ ]:
# =======================================================
# Cellule 4 : 👉 À exécuter uniquement si Cellule 2 OK
# =======================================================

from transformers import TrainingArguments

print("✅ TrainingArguments import OK")

In [ ]:
!git clone https://github.com/raym648/medical-triage-agent-ai-poc.git

In [ ]:
%cd /content/medical-triage-agent-ai-poc

In [ ]:
!ls

In [7]:
from pathlib import Path
import sys

PROJECT_ROOT=Path.cwd().resolve()
while PROJECT_ROOT!=PROJECT_ROOT.parent:
    if (PROJECT_ROOT/'backend').exists() and (PROJECT_ROOT/'frontend').exists():
        break
    PROJECT_ROOT=PROJECT_ROOT.parent
else:
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))


In [8]:
required=['backend/app/training/colab','backend/app/training/sft','backend/app/training/dpo','backend/app/training/evaluation']
missing=[p for p in required if not (PROJECT_ROOT/Path(p)).exists()]
if missing:
    raise FileNotFoundError(missing)


In [ ]:
# !grep -n "def get_training_arguments_precision\|def is_bf16_supported\|def is_fp16_supported" -A 15 backend/app/training/colab/colab_environment.py

In [ ]:
# !grep -n "dtype=None\|FastLanguageModel.from_pretrained" -B 3 -A 10 backend/app/training/shared/training_model_loader.py

In [ ]:
# print(model.dtype)  # doit afficher torch.float16 sur T4, plus jamais bfloat16

# Section 3


In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN_06")
if hf_token:
    os.environ["HF_TOKEN_06"] = hf_token

# try:
#     from google.colab import drive
#     drive.mount("/content/drive")
# except Exception:
#     pass

from backend.app.training.colab.colab_environment import log_environment_info
from backend.app.training.colab.colab_gpu_detector import log_gpu_info
from backend.app.training.colab.colab_hf_login import ensure_hf_login
from backend.app.training.colab.colab_checkpoint_sync import (
    create_default_checkpoint_sync,
)

environment = log_environment_info()
gpu_info = log_gpu_info()
hf_status = ensure_hf_login()

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="sft",
    # training_type=TRAINING_TYPE,
)

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="dpo",
    # training_type=TRAINING_TYPE,
)

checkpoint_sync.get_status()

print("✅ HF Login :", hf_status)

# Section 4


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.sft.train_sft',run_name='__main__')


# Section 5


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.dpo.train_dpo',run_name='__main__')



# Section 6


In [ ]:
import os
from google.colab import userdata
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')  # ← ligne ajoutée

import runpy
runpy.run_module('backend.app.training.evaluation.clinical_eval_runner',run_name='__main__')


# Section 7


In [ ]:
TRAINING_TYPE = "sft"  # ou "dpo"

from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="sft",
    # training_type=TRAINING_TYPE,
)

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="dpo",
    # training_type=TRAINING_TYPE,
)

print(checkpoint_sync.sync_latest_checkpoint())


# Section 8


In [ ]:
TRAINING_TYPE = "sft"  # ou "dpo"

from backend.app.training.colab.colab_checkpoint_sync import create_default_checkpoint_sync

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="sft",
    # training_type=TRAINING_TYPE,
)

checkpoint_sync = create_default_checkpoint_sync(
    output_dir="./outputs",
    training_type="dpo",
    # training_type=TRAINING_TYPE,
)

print(checkpoint_sync.get_status())